In [0]:
%run ../00-common/01.environment-config

In [0]:
control_path = "s3://bizf1project123/Control/Aaisha_increment/batch_control"

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
# Get batch folders from landing
landing_batches = sorted([
    file.name.rstrip("/")
    for file in dbutils.fs.ls(landing_folder_path)
    if file.isDir()
])

# Read tracked batches from the Delta control path
if DeltaTable.isDeltaTable(spark, control_path):
    tracked_batches = [
        row.batch_id
        for row in (
            spark.read.format("delta")
            .load(control_path)
            .filter(F.col("status").isin("in_progress", "completed"))
            .select("batch_id")
            .distinct()
            .collect()
        )
    ]
else:
    tracked_batches = []

# Identify the earliest unprocessed batch
new_batches = sorted(set(landing_batches) - set(tracked_batches))
next_batch = new_batches[0] if new_batches else None

print(f"Landing batches      : {landing_batches}")
print(f"Tracked batches      : {tracked_batches}")
print(f"Next batch to process: {next_batch}")

In [0]:
if next_batch is None:
    dbutils.jobs.taskValues.set(key="p_batch_id", value="")
    dbutils.jobs.taskValues.set(key="has_batch", value="false")
else:
    dbutils.jobs.taskValues.set(key="p_batch_id", value=next_batch)
    dbutils.jobs.taskValues.set(key="has_batch", value="true")